# 🎓 [Colab 실습] 지식증류 첫걸음 — 순수 파이썬으로 밑바닥부터

**온디바이스 AI 프로그래밍 · 프루닝&지식증류 미니랩 들어가기 전 준비 실습**

| 항목 | 내용 |
| --- | --- |
| 대상 | 지식증류(Knowledge Distillation)를 처음 접하는 분 |
| 도구 | **순수 파이썬 + numpy + matplotlib만** — PyTorch 없음! |
| 환경 | Google Colab CPU 런타임 |
| 진행 | 위에서부터 셀을 하나씩 실행 (`Shift + Enter`) |

## 이 실습의 목표

"큰 교사 모델의 지식을 작은 학생 모델에 옮긴다" — 말은 신비롭지만,
그 실체는 **정답표 대신 교사의 '확률 분포'를 베끼게 하는 것**뿐입니다.
이 실습에서는 그 확률 분포에 숨은 **어두운 지식(dark knowledge)**을 온도 다이얼로 직접 밝혀내고,
마지막에는 **오염된 정답표로 배우던 학생을 교사가 구해내는** 실험까지 수행합니다.

## 로드맵

| Part | 주제 | 핵심 발견 |
| --- | --- | --- |
| 1 | one-hot의 빈곤 — 정답표가 버리는 정보 | soft label의 존재 |
| 2 | 온도 T — 어두운 지식을 밝히는 다이얼 | softmax(z/T) |
| 3 | 따라하기 손실 — 분포 사이의 거리 | cross-entropy와 T² 보정 |
| 4 | KD 손실 조립 — α 혼합 | 완성된 레시피 |
| 5 | 🏁 종합: 교사가 학생을 구하다 | 오염 라벨 61% → KD 99% |
| 6 | 정리 — PyTorch 미니랩·NPU와의 연결 | 다음 실습 지도 |

> 💡 각 Step의 **✅ 확인**을 점검하고 **✏️ 직접 해보기**로 실험하세요.


---
# Part 0. 환경 준비

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
rng = np.random.default_rng(42)

try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"],
                   capture_output=True, timeout=120)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rc("font", family="NanumGothic")
    print("한글 폰트 설정 완료")
except Exception as e:
    print("한글 폰트 생략:", e)
plt.rc("axes", unicode_minus=False)
print("준비 완료 🎓")

---
# Part 1. one-hot의 빈곤 — 정답표가 버리는 정보

### Step 1-1. 애매한 이미지 하나를 만들어 봅시다

숫자 **7**인데 **1**을 닮은 이미지 — 현실 데이터엔 이런 경계 사례가 가득합니다.

In [ ]:
B0 = np.array([[1,1,1,1],[1,0,0,1],[1,0,0,1],[1,1,1,1]], float)   # 0
B1 = np.array([[0,0,1,0],[0,1,1,0],[0,0,1,0],[0,1,1,1]], float)   # 1
B7 = np.array([[1,1,1,1],[0,0,0,1],[0,0,1,0],[0,1,0,0]], float)   # 7

ambiguous = 0.55 * B7 + 0.45 * B1        # 7과 1의 중간쯤인 애매한 이미지

fig, axes = plt.subplots(1, 4, figsize=(8, 2.2))
for ax, (img, t) in zip(axes, [(B0,"확실한 0"), (B1,"확실한 1"), (B7,"확실한 7"), (ambiguous,"애매한 입력\n(7? 1?)")]):
    ax.imshow(img, cmap="gray"); ax.set_title(t, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

print("이 애매한 이미지의 정답 라벨(one-hot)은:  [0, 0, 1]   ← '무조건 7'")
print("사람이 보기엔:                          '7인데... 1을 좀 닮았네'")
print()
print("💡 one-hot 라벨은 '1과 닮았다'는 정보를 통째로 버립니다. 그 정보는 어디에 있을까요?")

### Step 1-2. 교사의 출력에는 그 정보가 살아있다

잘 학습된 큰 모델(교사)에 이 이미지를 넣으면 **logit**(점수)이 나옵니다.
예를 들어 이렇게요 (Part 5에서 진짜 교사를 학습해 확인할 값의 미리보기):

```text
z = [1.0, 6.0, 9.0]      # 클래스 [0, 1, 7]에 대한 점수
```

7이 1등이지만 **1의 점수도 꽤 높습니다** — "1과 닮았다"는 지식이 점수 차이에 인코딩되어 있습니다.
이 숨은 유사도 정보를 힌튼은 **어두운 지식(dark knowledge)**이라 불렀습니다.
문제는 이걸 확률로 바꾸는 순간(softmax) 거의 사라져 보인다는 것 — Part 2에서 밝혀냅니다.

In [ ]:
z_teacher = np.array([1.0, 6.0, 9.0])    # 교사의 logit (클래스 0/1/7)
classes = ["0", "1", "7"]

print("교사 logit:", dict(zip(classes, z_teacher)))
print()
print("점수 차이를 읽어보면:")
print("  '7'(9.0) vs '1'(6.0) → 차이 3.0  : 꽤 닮았다고 본다")
print("  '7'(9.0) vs '0'(1.0) → 차이 8.0  : 전혀 다르다고 본다")
print()
print("✅ 어두운 지식 = '오답들 사이의 순위와 간격' — one-hot에는 없는 정보입니다.")

> **✅ Part 1 확인**
> - [ ] one-hot 라벨이 클래스 간 유사도 정보를 버린다는 것을 예시로 설명할 수 있다
> - [ ] 교사 logit의 '오답 점수들'에 어두운 지식이 들어있음을 이해했다

---
# Part 2. 온도 T — 어두운 지식을 밝히는 다이얼

### Step 2-1. softmax에 온도 넣기 (순수 파이썬)

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

T=1이면 보통의 softmax. **T를 올리면 점수를 나눠서 차이가 줄어들고, 분포가 부드러워집니다.**

In [ ]:
def softmax_T(z, T=1.0):
    """온도 T를 가진 softmax — 순수 파이썬"""
    scaled = [zi / T for zi in z]
    m = max(scaled)                          # overflow 방지 (관례)
    exps = [math.exp(s - m) for s in scaled]
    total = sum(exps)
    return [e / total for e in exps]

p1 = softmax_T(z_teacher, T=1)
print("T=1 (보통 softmax):", [f"{p*100:5.1f}%" for p in p1])
print("→ '7'이 95%. '1과 닮았다'는 정보(1의 확률)가 4.7%로 짓눌려 거의 안 보입니다.")

### Step 2-2. 온도를 올려가며 관찰 — 어두운 지식이 드러나는 순간

> 📎 함께 보기: HTML 애니메이션 「지식증류 온도 다이얼」 — 같은 logit을 슬라이더로 조작합니다.

In [ ]:
temps = [1, 2, 3, 5, 10]
print(f"{'T':>4} | {'p(0)':>7} {'p(1)':>7} {'p(7)':>7} | 해석")
print("-" * 56)
rows = []
for T in temps:
    p = softmax_T(z_teacher, T)
    rows.append(p)
    note = "정답만 보임" if T == 1 else ("어두운 지식 뚜렷!" if T == 5 else "")
    print(f"{T:>4} | {p[0]*100:6.1f}% {p[1]*100:6.1f}% {p[2]*100:6.1f}% | {note}")

x = np.arange(3); w = 0.15
plt.figure(figsize=(8, 3.4))
for i, (T, p) in enumerate(zip(temps, rows)):
    plt.bar(x + (i-2)*w, p, w, label=f"T={T}")
plt.xticks(x, ["클래스 0", "클래스 1", "클래스 7"])
plt.ylabel("확률"); plt.title("온도가 오를수록 '1과의 유사도'가 드러난다")
plt.legend(); plt.grid(alpha=0.3, axis="y"); plt.tight_layout(); plt.show()

print(f"\nT=5에서 p(1) = {rows[3][1]*100:.0f}% — '7은 1과 닮았다'는 교사의 판단이 학생에게 전달 가능한 크기가 됐습니다.")

### Step 2-3. 왜 나누면 부드러워지나 + 극한 확인

지수함수는 **차이를 증폭**합니다. z를 T로 나누면 차이가 1/T로 줄어 증폭이 약해집니다.
극한도 확인해 봅시다: T→0이면 argmax(one-hot), T→∞면 균등분포.

In [ ]:
for T in [0.1, 1, 5, 100]:
    p = softmax_T(z_teacher, T)
    print(f"T={T:>5}: {[f'{v:.3f}' for v in p]}")
print()
print("✅ T는 'one-hot(정답만) ↔ 균등(정보 없음)' 사이를 잇는 다이얼 —")
print("   어두운 지식이 가장 잘 보이는 중간 온도(보통 3~10)를 골라 쓰는 것이 KD의 첫 번째 요령입니다.")

> **✅ Part 2 확인**
> - [ ] softmax_T를 순수 파이썬으로 구현했다
> - [ ] T=1에선 4.7%였던 p(1)이 T=5에서 31%로 드러남을 확인했다
> - [ ] T→0 / T→∞ 극한의 의미를 안다

---
# Part 3. 따라하기 손실 — 두 분포 사이의 거리

### Step 3-1. cross-entropy 구현 — 분포를 재는 자

학생이 교사를 '얼마나 못 따라하는지'를 재려면 두 확률 분포 사이의 거리가 필요합니다.

$$H(p^{teacher}, p^{student}) = -\sum_i p_i^{teacher} \log p_i^{student}$$

In [ ]:
def cross_entropy(p_target, p_pred):
    """타깃 분포 p_target 기준으로 p_pred가 얼마나 다른지 (작을수록 비슷)"""
    return -sum(t * math.log(max(q, 1e-12)) for t, q in zip(p_target, p_pred))

p_teacher = softmax_T(z_teacher, T=5)

# 학생 후보 3명: 교사와 다른 정도가 다름
students = {
    "교사와 똑같음":        p_teacher,
    "정답만 아는 학생":     [0.01, 0.01, 0.98],
    "아예 헛다리(0이 정답)": [0.90, 0.05, 0.05],
}
for name, p in students.items():
    print(f"{name:<16}: CE = {cross_entropy(p_teacher, p):.3f}")
print()
print("✅ 교사 분포와 가까울수록 손실이 작다 — 이 손실을 줄이는 방향으로 학생을 밀면 '따라하기'가 됩니다.")

### Step 3-2. 손실 지형 걸어보기 — 학생을 교사 쪽으로 보간

학생 logit을 교사 logit 쪽으로 조금씩 옮기며(보간) 손실이 매끄럽게 감소하는지 확인합니다.
학습이 이 경사면을 따라 굴러 내려가는 것과 같습니다.

In [ ]:
z_student0 = np.array([2.0, 1.0, 0.5])      # 헛배운 초기 학생

alphas = np.linspace(0, 1, 21)
losses = []
for a in alphas:
    z_mix = (1 - a) * z_student0 + a * z_teacher
    losses.append(cross_entropy(p_teacher, softmax_T(z_mix, T=5)))

plt.figure(figsize=(7, 3.2))
plt.plot(alphas, losses, "o-", lw=2)
plt.xlabel("학생 → 교사 보간 비율"); plt.ylabel("따라하기 손실 (T=5)")
plt.title("교사에게 다가갈수록 손실이 매끄럽게 감소")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"시작 손실 {losses[0]:.3f} → 도착 손실 {losses[-1]:.3f} (교사 분포의 자체 엔트로피)")

### Step 3-3. T² 보정 — 온도를 올리면 신호가 약해지는 문제

따라하기 손실의 기울기(gradient)는 다음과 같습니다:

$$\frac{\partial}{\partial z^{s}} H\big(p^{t}_{T},\ \mathrm{softmax}(z^{s}/T)\big) = \frac{1}{T}\big(p^{s}_{T} - p^{t}_{T}\big)$$

**T로 나눈 만큼 기울기가 작아집니다** — 게다가 분포 차이 $(p^s_T - p^t_T)$ 자체도 T가 크면 작아지죠.
그래서 손실에 $T^2$을 곱해 신호 크기를 되살립니다. 수치로 확인합시다.

In [ ]:
def kd_grad_norm(T, with_correction):
    ps = np.array(softmax_T(z_student0, T))
    pt = np.array(softmax_T(z_teacher, T))
    g = (ps - pt) / T
    if with_correction:
        g = g * T**2                          # T² 보정
    return float(np.abs(g).sum())

print(f"{'T':>4} | {'보정 없음':>10} | {'T² 보정':>10}")
for T in [1, 3, 5, 10]:
    print(f"{T:>4} | {kd_grad_norm(T, False):>10.4f} | {kd_grad_norm(T, True):>10.4f}")
print()
print("💡 보정 없이는 T=10에서 학습 신호가 사실상 소멸 — T² 보정 덕에 온도를 바꿔도")
print("   '따라하기'와 '정답 맞히기'(α 혼합의 상대) 사이 균형이 유지됩니다.")

> **✅ Part 3 확인**
> - [ ] cross-entropy를 구현했고 '교사와 가까울수록 작다'를 확인했다
> - [ ] 손실 지형이 교사 방향으로 매끄럽게 내려감을 봤다
> - [ ] T² 보정이 없으면 고온에서 학습 신호가 죽는 것을 수치로 확인했다

---
# Part 4. KD 손실 조립 — 완성된 레시피

부품이 다 모였습니다. 힌튼의 지식증류 손실:

$$L_{KD} = (1-\alpha)\cdot H\big(y^{onehot},\ p^{s}\big) \;+\; \alpha \cdot T^2 \cdot H\big(p^{t}_{T},\ p^{s}_{T}\big)$$

- 앞 항: **정답 맞히기** (기존 학습 그대로)
- 뒤 항: **교사 따라하기** (T로 어두운 지식을 밝히고, T²로 신호 보정)
- α: 두 선생님(정답표 vs 교사)의 발언권 배분 — 미니랩과 동일하게 **α=0.8** 사용

In [ ]:
def kd_loss(z_student, y_onehot, z_teacher, T=5.0, alpha=0.8):
    ps  = softmax_T(list(z_student), 1.0)
    psT = softmax_T(list(z_student), T)
    ptT = softmax_T(list(z_teacher), T)
    hard = cross_entropy(y_onehot, ps)
    soft = cross_entropy(ptT, psT)
    return (1 - alpha) * hard + alpha * T**2 * soft, hard, soft

y = [0, 0, 1]                                # 정답 '7'
total, hard, soft = kd_loss(z_student0, y, z_teacher)
print(f"정답 맞히기 항 : {hard:.3f}")
print(f"교사 따라하기 항: {soft:.3f} (T²={5**2} 배율 적용 전)")
print(f"KD 총손실 (α=0.8): {total:.3f}")
print()
print("✅ 레시피 완성 — Part 5에서 이 손실로 진짜 학생을 가르칩니다.")

---
# Part 5. 🏁 종합 — 교사가 학생을 구하다 (오염된 정답표 사건)

**시나리오**: 작은 학생 모델을 학습시켜 NPU에 올려야 합니다. 그런데 학생용 데이터가
겨우 60장인 데다, 라벨 작업 실수로 **정답표의 30%가 오염**되어 있습니다.
정답표만 믿고 배우면? 교사의 soft label을 함께 들으면? 직접 확인합시다.

### Step 5-1. 데이터 준비 — 오염된 정답표 만들기 (실행만 하면 됩니다)

In [ ]:
B0 = np.array([[1,1,1,1],[1,0,0,1],[1,0,0,1],[1,1,1,1]], float)
B1 = np.array([[0,0,1,0],[0,1,1,0],[0,0,1,0],[0,1,1,1]], float)
B7 = np.array([[1,1,1,1],[0,0,0,1],[0,0,1,0],[0,1,0,0]], float)
BASES = [B0, B1, B7]

rng = np.random.default_rng(42)               # 재현성
def softmax_np(z, T=1.0):
    z = z / T; z = z - z.max(-1, keepdims=True)
    e = np.exp(z); return e / e.sum(-1, keepdims=True)

def make_data(n, noise=0.55):
    X, y = [], []
    for _ in range(n):
        c = rng.integers(0, 3)
        X.append((BASES[c] + rng.normal(0, noise, (4,4))).reshape(-1)); y.append(c)
    return np.array(X), np.array(y)

Xtr, ytr = make_data(600)                     # 교사용: 풍부하고 깨끗함
Xte, yte = make_data(300)                     # 시험지

# 학생용: 60장뿐 + 라벨 30% 오염!
N_S = 60
Xs, ys_clean = Xtr[:N_S].copy(), ytr[:N_S].copy()
ys = ys_clean.copy()
n_bad = int(N_S * 0.3)
bad_idx = rng.choice(N_S, n_bad, replace=False)
for i in bad_idx:
    ys[i] = (ys[i] + rng.integers(1, 3)) % 3   # 무작위 다른 클래스로 오염

print(f"학생용 데이터 {N_S}장 중 {n_bad}장({n_bad/N_S*100:.0f}%)의 라벨이 오염됨")
print(f"예: 원래 정답 {ys_clean[bad_idx[:5]]} → 오염된 정답표 {ys[bad_idx[:5]]}")

### Step 5-2. 등장인물 준비 — 공용 학습 루프와 모델 (실행만 하면 됩니다)

교사(은닉 24)와 학생(은닉 4)은 같은 구조의 2층 신경망입니다. KD 기울기는 Part 3~4에서 유도한 식 그대로:
`(1−α)·(pˢ−y) + α·T·(pˢ_T − pᵗ_T)` (T²보정 × 1/T = T)

In [ ]:
def init_model(H, seed):
    r = np.random.default_rng(seed)
    return [r.normal(0, 0.3, (16, H)), np.zeros(H), r.normal(0, 0.3, (H, 3)), np.zeros(3)]

def forward(m, X):
    h = np.maximum(0, X @ m[0] + m[1])
    return h @ m[2] + m[3], h

def accuracy(m):
    return (forward(m, Xte)[0].argmax(1) == yte).mean() * 100

def train(m, X, y, epochs, z_teacher=None, T=5.0, alpha=0.8, lr=0.3):
    """z_teacher=None이면 보통 CE 학습, 주어지면 KD 학습"""
    pt = softmax_np(z_teacher, T) if z_teacher is not None else None
    for _ in range(epochs):
        z, h = forward(m, X)
        d = softmax_np(z) - np.eye(3)[y]                      # 정답 맞히기 신호
        if pt is not None:
            d = (1 - alpha) * d + alpha * T * (softmax_np(z, T) - pt)   # + 따라하기 신호
        d /= len(X)
        gW2 = h.T @ d; gb2 = d.sum(0)
        dh = d @ m[2].T * (h > 0)
        gW1 = X.T @ dh; gb1 = dh.sum(0)
        m[2] -= lr*gW2; m[3] -= lr*gb2; m[0] -= lr*gW1; m[1] -= lr*gb1
    return m

print("학습 루프 준비 완료 — CE와 KD가 'd' 한 줄 차이임을 눈여겨보세요!")

### Step 5-3. 교사 학습 — 깨끗한 600장으로

In [ ]:
teacher = train(init_model(24, seed=3), Xtr, ytr, epochs=400)
acc_teacher = accuracy(teacher)
n_teacher = 16*24+24+24*3+3
print(f"교사 (은닉 24, 파라미터 {n_teacher}개): 정확도 {acc_teacher:.1f}%")

# 학생용 60장에 대한 교사의 logit (soft label의 재료) 미리 계산
z_teacher_s = forward(teacher, Xs)[0]

# Part 1의 '애매한 이미지'에 대한 교사 분포 확인
z_amb = forward(teacher, (0.55*B7+0.45*B1).reshape(1,-1))[0][0]
print(f"\n애매한 7 이미지에 대한 교사 logit: {np.round(z_amb, 1)}")
print(f"T=5 soft label: {np.round(softmax_np(z_amb, 5)*100, 1)}%  ← Part 1~2의 예고가 진짜였습니다!")

### Step 5-4. 대결 — 정답표만 믿은 학생 vs 교사 말도 들은 학생

두 학생은 **완전히 같은 초기값, 같은 60장, 같은 오염된 정답표**로 시작합니다.
다른 것은 단 하나 — KD 학생은 교사의 soft label(T=5, α=0.8)을 함께 듣습니다.

In [ ]:
student_ce = train(init_model(4, seed=7), Xs, ys, epochs=300)
student_kd = train(init_model(4, seed=7), Xs, ys, epochs=300,
                   z_teacher=z_teacher_s, T=5.0, alpha=0.8)

acc_ce, acc_kd = accuracy(student_ce), accuracy(student_kd)
n_student = 16*4+4+4*3+3

print("════ 대결 결과 ════")
print(f"{'모델':<28}{'파라미터':>8}{'정확도':>9}")
print(f"{'교사 (은닉 24, 깨끗한 600장)':<25}{n_teacher:>8}{acc_teacher:>8.1f}%")
print(f"{'학생 CE만 (오염 정답표 60장)':<25}{n_student:>8}{acc_ce:>8.1f}%   ← 오염 라벨을 외워버림 💥")
print(f"{'학생 KD   (같은 조건 + 교사)':<25}{n_student:>8}{acc_kd:>8.1f}%   ← 교사가 오염을 교정 ✨")
print()
print(f"KD 이득: {acc_kd-acc_ce:+.1f}%p — 파라미터 {n_teacher/n_student:.0f}분의 1 학생이 교사({acc_teacher:.1f}%)에 필적!")
print()
print("💡 왜 구조가 가능한가: 오염 라벨은 '무조건 이 답'이라 우기지만,")
print("   교사 soft label은 이미지가 실제로 보여주는 것과 일치하는 부드러운 신호를 줍니다.")
print("   α=0.8이라 학생 귀에는 교사 목소리가 4배 크게 들리는 셈입니다.")

### Step 5-5. 온도 스윕 — 적정 온도가 존재한다

T를 바꿔가며 KD 학생의 정확도를 잽니다. 너무 낮으면(어두운 지식이 안 보임),
너무 높으면(균등분포에 가까워져 정보 소실) 모두 손해 — **종 모양**이 나옵니다.

In [ ]:
temps = [1, 2, 3, 5, 8, 12]
accs_T = []
for T in temps:
    m = train(init_model(4, seed=7), Xs, ys, epochs=300,
              z_teacher=z_teacher_s, T=float(T), alpha=0.8)
    accs_T.append(accuracy(m))
    print(f"T={T:>2} → KD 학생 정확도 {accs_T[-1]:.1f}%")

plt.figure(figsize=(7, 3.4))
plt.plot(temps, accs_T, "o-", lw=2, color="#e8a33d")
plt.axhline(acc_ce, ls="--", c="#c44e52", label=f"CE만 {acc_ce:.1f}%")
plt.axhline(acc_teacher, ls=":", c="gray", label=f"교사 {acc_teacher:.1f}%")
plt.xlabel("온도 T"); plt.ylabel("KD 학생 정확도 (%)")
plt.title("온도 스윕 — 어두운 지식이 가장 잘 전달되는 온도가 있다")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("💡 미니랩과 교안이 T=5를 쓰는 이유가 이 곡선의 정점입니다 (문제마다 정점은 다릅니다).")

### Step 5-6. 리포트 과제 — 직접 실험하고 표를 채우세요

| 실험 | 조건 | KD 정확도 | 관찰 |
| --- | --- | --- | --- |
| 기준 | T=5, α=0.8, 오염 30% | | |
| 실험1 | α 스윕: 0 / 0.3 / 0.5 / 0.8 / 1.0 | | |
| 실험2 | 오염률 0%로 (깨끗한 정답표) | | |
| 실험3 | 교사를 은닉 4짜리 '약한 교사'로 교체 | | |

**분석 질문 (2~3문장씩):**
1. 실험1에서 α=1.0(정답표 완전 무시)의 결과는? 오염 30% 상황에서 그것이 합리적인 이유 또는 위험한 이유는?
2. 실험2처럼 정답표가 깨끗하면 KD 이득이 줄어드나요? 그래도 KD를 쓰는 실전 이유(60장뿐이라는 조건)와 연결해 설명하세요.
3. 실험3의 '약한 교사'로도 이득이 있나요? "교사는 학생보다 얼마나 커야 하는가"에 대한 여러분의 가설은?

In [ ]:
# ✏️ 실험 공간 — 예시 (실험1의 일부):
for a in [0.0, 0.5, 1.0]:
    m = train(init_model(4, seed=7), Xs, ys, epochs=300,
              z_teacher=z_teacher_s, T=5.0, alpha=a)
    print(f"α={a:.1f} → {accuracy(m):.1f}%")
# 실험2·3은 직접 작성해 보세요!
# 힌트(실험2): ys 대신 ys_clean 사용 / 힌트(실험3): weak = train(init_model(4, seed=11), Xtr, ytr, 400)

---
# Part 6. 정리 — 오늘 만든 부품 ↔ PyTorch 미니랩

| 오늘 직접 만든 것 | 프루닝&지식증류 미니랩 / 실전에서의 이름 |
| --- | --- |
| `softmax_T(z, T)` | `F.softmax(logits / T)` — 온도 스케일링 |
| 어두운 지식 관찰 (T=1→5) | soft label 시각화 (미니랩의 온도별 그래프) |
| `cross_entropy(p_t, p_s)` | `KLDivLoss` (상수 차이만 있음 — 기울기는 동일) |
| T² 보정 실험 | KD 손실의 `* T * T` 항 |
| `kd_loss` (α 혼합) | `L = α·KL(T)·T² + (1−α)·CE` — 미니랩과 동일 레시피 (T=5, α=0.8) |
| 오염 정답표 실험 | KD의 정규화·라벨 노이즈 내성 (실전 KD가 사랑받는 이유 중 하나) |
| 60장 소량 데이터 조건 | 데이터 효율 — 교사의 soft label이 '추가 감독 신호' 역할 |

## 왜 작동하는가 — 세 문장 요약

1. **정보량**: soft label은 클래스당 확률 하나가 아니라 '오답들의 순위·간격'까지 담아 라벨 하나보다 훨씬 많은 것을 가르친다.
2. **교정**: 정답표가 틀렸거나 데이터가 애매할 때, 교사의 분포는 이미지가 실제로 보여주는 것과 일치하는 신호를 준다.
3. **온도**: 그 정보는 T를 올려야 보이고, T²보정과 α로 정답 신호와 균형을 맞춘다.

## NPU와의 연결 (교안 Day 2)

- 온디바이스의 숙명: **NPU에는 작은 모델만** 올라갑니다(BlackSwan ≈30MB). KD는 "작게 만들되 실력은 큰 모델에게 물려받는" 표준 경로입니다.
- **트리플 콤보**가 교안의 최종 레시피: 지식증류(작은 모델을 똑똑하게) → 구조적 프루닝(더 작게) → INT8 양자화(4배 더 작게) — 「프루닝 첫걸음」 도전과제 3과 이어집니다.

## ✏️ 심화 도전 과제 (선택)

1. **자기증류(self-distillation)**: 학생과 같은 크기의 모델을 교사로 써도 이득이 있는지 실험
2. **앙상블 교사**: 시드가 다른 교사 3명의 soft label 평균으로 가르치면 단일 교사보다 나은지 실험
3. **트리플 콤보 완성**: KD로 학습한 은닉 4 학생을 「프루닝 첫걸음」의 연쇄 수술로 은닉 2로 줄이고,
   「양자화 첫걸음」의 `quantize()`로 INT8화 — 최종 압축률과 정확도를 리포트
4. **feature 증류 맛보기**: 출력 확률 대신 은닉층 h를 따라하게 하면(정렬용 행렬 필요) 어떻게 되는지 실험

---

수고하셨습니다! 🎉 이제 지식증류를 "큰 모델의 영혼을 옮기는 마법"이 아니라
**"온도로 밝힌 확률 분포를 α만큼 섞어 베끼게 하는 산수"**로 이해하게 되었습니다.
다음 실습 「프루닝&지식증류 미니랩」에서 같은 레시피가 CNN 규모로 펼쳐집니다.
